###Ingest Data from Silver Layer

In [0]:
df_stat_op= spark.read.table("e_commerce_catalog.silver_schema.orders_cleaned_table")

In [0]:
df_stat_op.display()

customer_id,final_amount,ProductCategory,Order_Date,State,MembershipTier,PaymentMethod
CUST-05985,218.07,Automotive,2024-10-20,KY,Bronze,Buy Now Pay Later
CUST-06197,105.7,Toys & Games,2024-04-16,MI,Gold,Apple Pay
CUST-06403,726.06,Electronics,2023-07-30,WV,Gold,Apple Pay
CUST-08146,331.12,Home & Kitchen,2023-01-10,IN,Silver,PayPal
CUST-07549,79.65,Toys & Games,2024-07-24,NV,Silver,PayPal
CUST-07139,2441.3,Electronics,2022-08-07,IA,Platinum,Google Pay
CUST-06698,161.06,Sports & Outdoors,2023-07-04,RI,Platinum,Apple Pay
CUST-02401,63.09,Clothing,2023-09-17,VA,Silver,Gift Card
CUST-04321,179.69,Automotive,2022-12-04,LA,Platinum,Apple Pay
CUST-07333,73.02,Automotive,2022-04-02,KS,Platinum,Gift Card


##Statistical Operations

### Mean


In [0]:
from pyspark.sql.functions import avg

df_stat_op.select(avg("final_amount").alias("mean_amount")).show()

+-----------------+
|      mean_amount|
+-----------------+
|320.4012312312307|
+-----------------+



###Median

In [0]:
from pyspark.sql.functions import expr

df_stat_op.select(expr("percentile_approx(final_amount, 0.5)").alias("median")).show()

+------+
|median|
+------+
|162.53|
+------+



###Mode

In [0]:
df_stat_op.groupBy("final_amount") \
  .count() \
  .orderBy("count", ascending=False) \
  .show(1)

+------------+-----+
|final_amount|count|
+------------+-----+
|      159.72|    8|
+------------+-----+
only showing top 1 row


###Variance & Standard Deviation

In [0]:
from pyspark.sql.functions import variance, stddev

df_stat_op.select(
    variance("final_amount").alias("variance"),
    stddev("final_amount").alias("std_dev")
).show()

+------------------+-----------------+
|          variance|          std_dev|
+------------------+-----------------+
|202279.38968892855|449.7548106345596|
+------------------+-----------------+



###Summary

In [0]:
df_stat_op.describe(["final_amount"]).show()

+-------+-----------------+
|summary|     final_amount|
+-------+-----------------+
|  count|             1665|
|   mean|320.4012312312307|
| stddev|449.7548106345596|
|    min|            50.07|
|    max|           5994.2|
+-------+-----------------+



### Skewness & Kurtosis (Distribution Shape)

In [0]:
from pyspark.sql.functions import skewness, kurtosis

df_stat_op.select(
    skewness("final_amount").alias("skewness"),
    kurtosis("final_amount").alias("kurtosis")
).show()

+-----------------+------------------+
|         skewness|          kurtosis|
+-----------------+------------------+
|4.745303243575731|35.775468638347625|
+-----------------+------------------+



###Interquartile Range (IQR) → Outlier Detection

In [0]:
quantiles = df_stat_op.selectExpr(
    "percentile_approx(final_amount, 0.25) as Q1",
    "percentile_approx(final_amount, 0.75) as Q3"
).collect()[0]

Q1 = quantiles["Q1"]
Q3 = quantiles["Q3"]
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = df_stat_op.filter((df_stat_op.final_amount < lower_bound) | (df_stat_op.final_amount > upper_bound))
outliers.show()

+-----------+------------+-----------------+----------+-----+--------------+-----------------+
|customer_id|final_amount|  ProductCategory|Order_Date|State|MembershipTier|    PaymentMethod|
+-----------+------------+-----------------+----------+-----+--------------+-----------------+
| CUST-07139|      2441.3|      Electronics|2022-08-07|   IA|      Platinum|       Google Pay|
| CUST-01669|     1033.88|      Electronics|2024-04-13|   WA|          Gold|           PayPal|
| CUST-03864|     2406.24|      Electronics|2023-11-13|   PA|      Platinum|        Gift Card|
| CUST-07100|     2002.94|      Electronics|2022-03-06|   NC|        Silver|        Apple Pay|
| CUST-01828|      998.77|      Electronics|2022-06-04|   CT|      Platinum|       Google Pay|
| CUST-03683|      750.33|      Electronics|2023-03-05|   WI|          Gold|       Google Pay|
| CUST-04136|     2553.82|      Electronics|2024-08-09|   NJ|        Silver|        Gift Card|
| CUST-06429|      992.83|      Electronics|2024-0

###Z Score

In [0]:
from pyspark.sql.functions import mean, stddev, col

stats = df_stat_op.select(
    mean("final_amount").alias("mean"),
    stddev("final_amount").alias("std")
).collect()[0]

mean_val = stats["mean"]
std_val = stats["std"]

df_z = df_stat_op.withColumn(
    "z_score",
    (col("final_amount") - mean_val) / std_val
)

df_z.display()

customer_id,final_amount,ProductCategory,Order_Date,State,MembershipTier,PaymentMethod,z_score
CUST-05985,218.07,Automotive,2024-10-20,KY,Bronze,Buy Now Pay Later,-0.22752670746723416
CUST-06197,105.7,Toys & Games,2024-04-16,MI,Gold,Apple Pay,-0.4773739516611474
CUST-06403,726.06,Electronics,2023-07-30,WV,Gold,Apple Pay,0.9019553747439072
CUST-08146,331.12,Home & Kitchen,2023-01-10,IN,Silver,PayPal,0.02383247163859383
CUST-07549,79.65,Toys & Games,2024-07-24,NV,Silver,PayPal,-0.5352943993896464
CUST-07139,2441.3,Electronics,2022-08-07,IA,Platinum,Google Pay,4.715677784027237
CUST-06698,161.06,Sports & Outdoors,2023-07-04,RI,Platinum,Apple Pay,-0.3542846623617344
CUST-02401,63.09,Clothing,2023-09-17,VA,Silver,Gift Card,-0.5721144613621587
CUST-04321,179.69,Automotive,2022-12-04,LA,Platinum,Apple Pay,-0.3128620926426581
CUST-07333,73.02,Automotive,2022-04-02,KS,Platinum,Gift Card,-0.5500357647808153


###Correlation Matrix

In [0]:
from pyspark.sql.functions import month

df_corr = df_stat_op.withColumn("month", month("Order_Date"))

df_corr.stat.corr("final_amount", "month")

0.030471250179123856

###Covariance

In [0]:
df_corr.stat.cov("final_amount", "month")

47.48899090797532

###1. ANOVA using ProductCategory

In [0]:
# Convert to Pandas
pdf = df_stat_op.select("final_amount", "ProductCategory").toPandas()

# ANOVA
from scipy.stats import f_oneway

groups = [
    group["final_amount"].values
    for name, group in pdf.groupby("ProductCategory")
]

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 108.08970410458927
P-value: 3.05504559197993e-159


###2. ANOVA using MembershipTier

In [0]:
pdf = df_stat_op.select("final_amount", "MembershipTier").toPandas()

groups = [
    group["final_amount"].values
    for name, group in pdf.groupby("MembershipTier")
]

f_stat, p_value = f_oneway(*groups)

print("F-statistic:", f_stat)
print("P-value:", p_value)

F-statistic: 0.7063233259952051
P-value: 0.5482281383912732


###ANOVA Table

In [0]:
%pip install statsmodels

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
dbutils.library.restartPython()

In [0]:
pdf = df_stat_op.select("final_amount", "ProductCategory").toPandas()

In [0]:
import statsmodels.api as sm
from statsmodels.formula.api import ols

model = ols('final_amount ~ C(ProductCategory)', data=pdf).fit()

anova_table = sm.stats.anova_lm(model, typ=2)
print(anova_table)

                          sum_sq      df           F         PR(>F)
C(ProductCategory)  1.246058e+08     9.0  108.089704  3.055046e-159
Residual            2.119871e+08  1655.0         NaN            NaN
